# verify_dbms — LLaVA 13B (Local Ollama)

Sends each question from `dbms_dataset_mixed.json` to **llava:13b** running via Ollama inside Colab.

- Handles all question types: Multiple Choice, Short Answer, Explanation, Analysis, SQL Interpretation, etc.
- Uploads both the JSON and an `images/` zip containing PNG diagrams
- Answers written to `/content/outputs/dbms/<id>.txt`
- Already-answered entries are skipped on re-run

> **Runtime:** GPU (T4 or better) recommended.

In [ ]:
!pip install -q ollama

In [ ]:
import subprocess, time, os

!curl -fsSL https://ollama.com/install.sh | sh

proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("Ollama service started.")

!ollama pull llava:13b
print("Model ready.")

In [ ]:
# Upload dbms_dataset_mixed.json
from google.colab import files
print("Upload dbms_dataset_mixed.json")
uploaded = files.upload()
JSON_PATH = list(uploaded.keys())[0]
print("JSON:", JSON_PATH)

In [ ]:
# Upload images as a zip (zip the 'images/' folder before uploading)
# Expected structure inside zip: image_001.png, image_002.png, ...
import zipfile

IMAGES_DIR = "/content/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

print("Upload images.zip (zip your 'images/' folder)")
img_uploaded = files.upload()
zip_name = list(img_uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(IMAGES_DIR)

print(f"Extracted images to {IMAGES_DIR}")
print("Sample files:", os.listdir(IMAGES_DIR)[:5])

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL           = "llava:13b"
OUTPUT_DIR      = "/content/outputs/dbms"
POST_CALL_DELAY = 3

SYSTEM_PROMPT = (
    "You are an expert database systems tutor. "
    "Analyze any provided diagram or image carefully before answering. "
    "Answer step by step, showing your reasoning clearly. "
    "For multiple-choice questions, state the correct option letter and explain why."
)

# Filter: set to None to run all, or e.g. ["dbms_001"] for specific ids
FILTER_IDS = None
# Filter: set to a string to run only one question type, e.g. "Multiple Choice"
FILTER_TYPE = None
# Filter: set to integer to run only last N entries
LAST_N = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json, time
from ollama import Client

client = Client(host="http://localhost:11434")

def load_entries(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def build_prompt(entry):
    q_type = entry.get("question_type", "")
    lines  = []
    if q_type:
        lines.append(f"[Question Type: {q_type}]")
        lines.append("")
    lines.append(entry["question"].strip())
    choices = entry.get("choices") or {}
    if choices:
        lines.append("")
        lines.append("Choices:")
        for letter, text in sorted(choices.items()):
            lines.append(f"  {letter}. {text}")
    return "\n".join(lines)

def resolve_image(entry):
    rel = entry.get("image", "")
    if not rel:
        return None
    # The JSON stores e.g. "images/image_001.png"; we extracted to /content/images/
    filename = os.path.basename(rel)
    full = os.path.join(IMAGES_DIR, filename)
    return full if os.path.exists(full) else None

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    prompt = build_prompt(entry)
    user_msg = {"role": "user", "content": prompt}

    img_path = resolve_image(entry)
    if img_path:
        user_msg["images"] = [img_path]
    elif entry.get("image"):
        print(f"  [WARN] #{entry['id']} — image not found: {entry['image']}")

    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            user_msg,
        ],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)

    q_type = entry.get("question_type", "")
    print(f"  [OK]   #{entry['id']} [{q_type}] → {out_path}")

print("Helpers defined.")

In [ ]:
# Load and filter entries
all_entries = load_entries(JSON_PATH)

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

if FILTER_TYPE:
    entries = [e for e in entries if e.get("question_type", "").lower() == FILTER_TYPE.lower()]

print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
# Run
for i, entry in enumerate(entries):
    try:
        ask(entry)
    except Exception as exc:
        print(f"  [ERR]  #{entry['id']} — {exc}")
    if i < len(entries) - 1:
        time.sleep(POST_CALL_DELAY)

print("\nDone.")

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/dbms_llava_answers"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")